# Week 3: Agribusiness Exploratory Data Analysis (EDA)
## Notebook 01: Agricultural Production, Regional Yield & Market Price Analytics

**Project Title**: Agribusiness Exploratory Data Analysis (EDA)  
**Role**: Junior Data Scientist Intern  
**Current Phase**: Week 3 – Exploratory Data Analysis (EDA) Report Design  

> **CRITICAL METHODOLOGICAL DISCLAIMER**  
> All data utilized in this demonstration notebook is **HYPOTHETICAL / SYNTHETIC**, designed strictly to validate exploratory algorithms, statistical profiling functions, and visualization routines. In accordance with internship requirements, no real-world empirical findings or statistics are claimed.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure repository src modules are discoverable
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.eda import (
    compute_numerical_profile,
    compute_categorical_profile,
    calculate_regional_share,
    calculate_spatial_yield_index,
    calculate_price_metrics,
    compute_seasonal_price_index,
    compute_correlation_matrices,
    detect_statistical_anomalies
)
from src.visualisation import (
    set_agri_style,
    plot_time_series_trend,
    plot_crop_comparison_bar,
    plot_distribution_histogram,
    plot_yield_boxplots,
    plot_bivariate_scatter,
    plot_correlation_heatmap
)
from src.utils import setup_logger

logger = setup_logger()
set_agri_style()
print("EDA environment successfully initialized at:", repo_root)

### 1. Generation of Representative Hypothetical Agribusiness Panel
We instantiate a structured **HYPOTHETICAL** agricultural panel covering:
- **5 Major Producing States**: Punjab, Haryana, Uttar Pradesh, Madhya Pradesh, Maharashtra
- **4 Staple Commodities**: Wheat, Rice, Chickpea, Mustard
- **Temporal Horizon**: Multi-year monthly observations (2018–2022)
- **Key Attributes**: Area (ha), Production (tonnes), Derived Yield (kg/ha), Wholesale Modal Price (INR/q)

In [ ]:
np.random.seed(42)

states = ["Punjab", "Haryana", "Uttar Pradesh", "Madhya Pradesh", "Maharashtra"]
crops = ["Wheat", "Rice", "Chickpea", "Mustard"]
dates = pd.date_range(start="2018-01-01", end="2022-12-31", freq="ME")

records = []
for dt in dates:
    for state in states:
        for crop in crops:
            # Assign realistic hypothetical baselines
            base_area = np.random.uniform(500, 2500)
            if crop == "Wheat":
                base_yield = np.random.normal(4200, 450) if state in ["Punjab", "Haryana"] else np.random.normal(3100, 400)
                base_price = 1900 + (dt.year - 2018) * 85 + np.random.normal(0, 45)
            elif crop == "Rice":
                base_yield = np.random.normal(3800, 350) if state in ["Punjab", "Haryana"] else np.random.normal(2600, 300)
                base_price = 1800 + (dt.year - 2018) * 75 + np.random.normal(0, 50)
            elif crop == "Chickpea":
                base_yield = np.random.normal(1200, 180)
                base_price = 4500 + (dt.year - 2018) * 120 + np.random.normal(0, 120)
            else:  # Mustard
                base_yield = np.random.normal(1500, 220)
                base_price = 4200 + (dt.year - 2018) * 150 + np.random.normal(0, 140)

            base_yield = max(300.0, base_yield)
            prod_tonnes = (base_area * base_yield) / 1000.0

            records.append({
                "date": dt,
                "year": dt.year,
                "month": dt.month,
                "state_name": state,
                "crop": crop,
                "crop_category": "Cereals" if crop in ["Wheat", "Rice"] else ("Pulses" if crop == "Chickpea" else "Oilseeds"),
                "area_ha": round(base_area, 2),
                "production_tonnes": round(prod_tonnes, 2),
                "yield_kg_per_ha": round(base_yield, 2),
                "modal_price": round(base_price, 2)
            })

df_hypothetical = pd.DataFrame(records)
print("Generated Hypothetical Agribusiness Panel (Rows, Cols):", df_hypothetical.shape)
display(df_hypothetical.head())

### 2. Univariate Statistical Profiling
We compute parametric and non-parametric descriptive statistics across area, production, yield, and commodity prices.

In [ ]:
numeric_profile = compute_numerical_profile(
    df_hypothetical, 
    ["area_ha", "production_tonnes", "yield_kg_per_ha", "modal_price"]
)
print("=== Numerical Profile (Parametric & Non-Parametric) ===")
display(numeric_profile)

In [ ]:
categorical_profile = compute_categorical_profile(
    df_hypothetical,
    ["state_name", "crop", "crop_category"]
)
print("=== Categorical Cardinality & Frequency Summary ===")
display(categorical_profile)

### 3. Production & Regional Yield Analysis
We examine regional output shares and compute the Spatial Yield Index ($SYI$) to benchmark state-level productivity.

In [ ]:
# Regional production share for Wheat
wheat_panel = df_hypothetical[df_hypothetical["crop"] == "Wheat"]
wheat_share = calculate_regional_share(wheat_panel, group_col="state_name", value_col="production_tonnes")
print("=== Wheat Regional Output Contribution ===")
display(wheat_share)

# Spatial Yield Index for Wheat across states
wheat_syi = calculate_spatial_yield_index(wheat_panel, crop_col="crop", region_col="state_name", yield_col="yield_kg_per_ha")
print("=== Spatial Yield Index (SYI) relative to National Mean (100.0) ===")
display(wheat_syi[["State", "yield_kg_per_ha", "National_Mean_Yield", "Spatial_Yield_Index"]].rename(columns={"State": "State_Name"}))

### 4. Visual Analytics: Distributions, Yield Spreads & Rankings

In [ ]:
# Plot 1: Yield Distribution Histogram with KDE
fig1 = plot_distribution_histogram(
    df_hypothetical,
    col="yield_kg_per_ha",
    title="[Hypothetical] Crop Yield Distribution Profile (kg/ha)",
    xlabel="Yield (kg/ha)"
)
plt.show()

In [ ]:
# Plot 2: Yield Boxplot by Crop Category
fig2 = plot_yield_boxplots(
    df_hypothetical,
    category_col="crop",
    value_col="yield_kg_per_ha",
    title="[Hypothetical] Crop Productivity Dispersion & Outlier Fences"
)
plt.show()

In [ ]:
# Plot 3: Bivariate Scatter: Cultivated Area vs Aggregate Production
fig3 = plot_bivariate_scatter(
    df_hypothetical,
    x_col="area_ha",
    y_col="production_tonnes",
    title="[Hypothetical] Cultivated Area vs. Production Volume (OLS Fit)",
    xlabel="Cultivated Area (ha)",
    ylabel="Production Volume (Metric Tonnes)"
)
plt.show()

### 5. Correlation Analysis: Pearson (Linear) vs. Spearman (Rank)
Evaluating statistical association across numerical features without claiming causation.

In [ ]:
corr_results = compute_correlation_matrices(
    df_hypothetical,
    ["area_ha", "production_tonnes", "yield_kg_per_ha", "modal_price"]
)

fig4 = plot_correlation_heatmap(
    corr_results["pearson"],
    title="[Hypothetical] Pearson Correlation Heatmap (Parametric Linear)"
)
plt.show()

print("Spearman Rank Correlation (Non-Parametric Monotonic):")
display(corr_results["spearman"])

### 6. Anomaly Investigation: Dual-Hurdle IQR & Z-Score
Diagnosing extreme observations to distinguish between data errors and genuine market/climatic shocks.

In [ ]:
anomaly_df = detect_statistical_anomalies(
    df_hypothetical,
    target_col="yield_kg_per_ha",
    group_col="crop",
    iqr_multiplier=3.0,
    z_threshold=3.0
)

iqr_count = anomaly_df["is_iqr_anomaly"].sum()
z_count = anomaly_df["is_zscore_anomaly"].sum()
print(f"Statistical Anomaly Audit: {iqr_count} IQR outliers, {z_count} Z-score violations detected.")

### 7. Key Findings & Week 4 Forecasting Handoff
- **Productivity Profiling**: Area-to-production relationships display strong positive linearity ($r > 0.85$), confirming that area remains a dominant baseline feature for forecasting.
- **Spatial Variation**: Spatial Yield Indices isolate high-efficiency agrarian states (Punjab/Haryana) from developing regions, suggesting fixed-effects panel models for econometric forecasting.
- **Price Volatility**: Non-linear price spikes in pulses indicate the need for seasonal differencing and autoregressive terms (SARIMA) in Week 4 modeling.